# PCAT-SLT on Kaggle

Start with **Settings → Accelerator → GPU T4 ×2**. This trainer uses GPU 0; TPU/DDP are not implemented. Import the source release as a separate input. Read `docs/KAGGLE_RUNBOOK.md` and `TRAINING_AND_DATASETS.md` in the source folder.

**Diagnostic mode generates toy geometry, not signs.** Real modes require authorized, already converted NPZ/JSONL. This notebook does not download/convert restricted corpora. It includes complete-epoch recovery and an output bundle. Its diagnostic is verified locally on CPU; Kaggle/CUDA and real-corpus performance remain unmeasured.


In [ ]:
from pathlib import Path

MODE = 'diagnostic'       # diagnostic, pose, visual
DEVICE = 'cuda'           # cpu is permitted only for the diagnostic
RESUME = False
STOP_AFTER_EPOCH = 1      # after epoch 1, restore + set RESUME=True and this to 2
RUN_NAME = None           # defaults: kaggle_demo, kaggle_pose, kaggle_visual
PREPARED_ROOT = None      # actual folder containing train.jsonl, val.jsonl, test.jsonl; auto if unique
SOURCE_ZIP = None         # explicit source ZIP path if automatic discovery is ambiguous
RESTORE_RUN_DIR = None    # actual saved run directory containing last.pt; auto if unique
RUN_GENERATION = True
EVAL_SPLIT = 'val'        # reserve test for a frozen final experiment
WORK_ROOT = Path('/kaggle/working')
INPUT_ROOT = Path('/kaggle/input')
PROJECT_ROOT = WORK_ROOT / 'PCAT-SLT'

assert MODE in {'diagnostic', 'pose', 'visual'}
assert DEVICE in {'cpu', 'cuda'}
assert DEVICE == 'cuda' or MODE == 'diagnostic'
assert type(STOP_AFTER_EPOCH) is int and STOP_AFTER_EPOCH >= 1
assert EVAL_SPLIT in {'val', 'test'}
RUN_NAME = RUN_NAME or {'diagnostic':'kaggle_demo', 'pose':'kaggle_pose', 'visual':'kaggle_visual'}[MODE]
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in {'.','..'}
print({'mode': MODE, 'device': DEVICE, 'resume': RESUME, 'run_name': RUN_NAME})


## Locate and verify source

The input may contain `PCAT-SLT.zip` or an extracted source tree. This cell preserves an existing workspace. If several releases are attached, choose `SOURCE_ZIP` explicitly. Only the project's source ZIP is extracted, not arbitrary video archives.


In [ ]:
import hashlib
import json
import os
import shutil
import zipfile

WORK_ROOT.mkdir(parents=True, exist_ok=True)
if not PROJECT_ROOT.exists():
    directories = sorted({p.parent for p in INPUT_ROOT.rglob('pyproject.toml')
                          if p.parent.name == 'PCAT-SLT'}) if INPUT_ROOT.exists() else []
    archives = [Path(SOURCE_ZIP)] if SOURCE_ZIP else sorted(INPUT_ROOT.rglob('PCAT-SLT.zip'))
    if SOURCE_ZIP or (not directories and len(archives) == 1):
        if len(archives) != 1:
            raise RuntimeError(f'Set SOURCE_ZIP to one source release; found: {archives}')
        with zipfile.ZipFile(archives[0]) as z:
            if z.testzip() is not None:
                raise RuntimeError('Source ZIP integrity check failed')
            for info in z.infolist():
                parts = Path(info.filename).parts
                if not parts or parts[0] != 'PCAT-SLT' or '..' in parts or Path(info.filename).is_absolute():
                    raise ValueError(f'Unexpected source archive member: {info.filename}')
                if (info.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError('Source ZIP must not contain symlinks')
            manifest = json.loads(z.read('PCAT-SLT/MANIFEST.sha256.json'))
            for name, digest in manifest.items():
                if hashlib.sha256(z.read('PCAT-SLT/' + name)).hexdigest() != digest:
                    raise RuntimeError(f'Source hash mismatch: {name}')
            z.extractall(WORK_ROOT)
        print('Verified and extracted', archives[0])
    elif len(directories) == 1:
        shutil.copytree(directories[0], PROJECT_ROOT,
                        ignore=shutil.ignore_patterns('__pycache__', '.pytest_cache', '.ruff_cache', '.venv', '.git'))
        print('Copied extracted source', directories[0])
    else:
        raise RuntimeError(f'Attach one source release or set SOURCE_ZIP. Folders: {directories}; ZIPs: {archives}')
if not (PROJECT_ROOT / 'src/pcat_slt/train.py').is_file():
    raise RuntimeError('Workspace is not a complete PCAT-SLT source tree')
os.chdir(PROJECT_ROOT)
for directory in ('reports', 'configs', 'data', 'runs', 'evaluation', 'releases'):
    (PROJECT_ROOT / directory).mkdir(exist_ok=True)
print('Project:', PROJECT_ROOT)


## Install and inspect the accelerator

Internet is needed for uncached packages; visual mode also needs the selected pretrained assets. Keep Kaggle's compatible CUDA torch rather than forcing the locally recorded CPU environment. Save these reports. CPU diagnostic success is not a CUDA test.


In [ ]:
import importlib
import platform
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev,multimodal]'], check=True)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
torch = importlib.import_module('torch')
if DEVICE == 'cuda':
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA unavailable. Select GPU T4 x2 and inspect the session before proceeding.')
    torch.cuda.set_device(0)
    gpu_output = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=True).stdout
    print(gpu_output)
    (PROJECT_ROOT / 'reports/kaggle_nvidia_smi.txt').write_text(gpu_output)
else:
    torch.set_num_threads(1)
environment = {'python': platform.python_version(), 'torch': str(torch.__version__),
               'cuda_runtime': torch.version.cuda, 'cuda_available': torch.cuda.is_available(),
               'device_count': torch.cuda.device_count(), 'requested_device': DEVICE,
               'gpu_0': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
(PROJECT_ROOT / 'reports/kaggle_environment.json').write_text(json.dumps(environment, indent=2)+'\n')
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True, check=True)
(PROJECT_ROOT / 'reports/kaggle_environment.txt').write_text(freeze.stdout)
print(environment)


## Run software checks

These tests include local random miniature pretrained fixtures, not real pretrained SLT quality. Stop and investigate failures rather than ignoring them.


In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=PROJECT_ROOT, check=True)


## Restore a previous run, when requested

For a fresh Kaggle session attach the **extracted** saved run bundle plus the same source release/data. `RESTORE_RUN_DIR` means the directory containing last.pt. Restore precedes diagnostic generation so the original NPZ bytes can be preserved. Nothing is deleted to make a fresh run fit.


In [ ]:
OUTPUT_DIR = PROJECT_ROOT / 'runs' / RUN_NAME
if RESUME:
    if not (OUTPUT_DIR / 'last.pt').exists():
        candidates = [Path(RESTORE_RUN_DIR)] if RESTORE_RUN_DIR else sorted({
            p.parent for p in INPUT_ROOT.rglob('last.pt') if p.parent.name == RUN_NAME
        })
        if len(candidates) != 1:
            raise RuntimeError(f'Attach one extracted saved run or set RESTORE_RUN_DIR: {candidates}')
        restored_run = candidates[0]
        if not (restored_run / 'last.pt').is_file():
            raise FileNotFoundError(restored_run / 'last.pt')
        if OUTPUT_DIR.exists():
            raise FileExistsError('Partial output exists; inspect it before restoring into a fresh run location')
        shutil.copytree(restored_run, OUTPUT_DIR)
        bundle_root = restored_run.parent.parent
        if MODE == 'diagnostic':
            saved_data = bundle_root / 'data/kaggle_demo'
            if not saved_data.is_dir():
                raise FileNotFoundError('Diagnostic recovery requires original data/kaggle_demo from the run bundle')
            target = PROJECT_ROOT / 'data/kaggle_demo'
            if not target.exists():
                shutil.copytree(saved_data, target)
        saved_config = bundle_root / f'configs/kaggle_runtime_{MODE}.yaml'
        target_config = PROJECT_ROOT / f'configs/kaggle_runtime_{MODE}.yaml'
        if saved_config.is_file() and not target_config.exists():
            shutil.copy2(saved_config, target_config)
    print('Resume checkpoint:', OUTPUT_DIR / 'last.pt')
elif (OUTPUT_DIR / 'last.pt').exists():
    raise FileExistsError('Run already has last.pt. Choose RESUME=True or a new RUN_NAME; do not overwrite it.')


## Prepare the runtime configuration

Diagnostic mode generates the small native pose task. Real modes require a prepared directory with all three manifests. Existing pose/video archives do not satisfy that requirement until converted. Relative observation paths are resolved against their original manifest before writing stable project-local manifests. There is no random resplit or silent truncation.


In [ ]:
import yaml
from pcat_slt.config import from_dict
from pcat_slt.data import read_manifest
from pcat_slt.synthetic import make_synthetic

def write_stable(path, content):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.read_text(encoding='utf-8') != content:
        raise RuntimeError(f'Existing file differs: {path}. Inspect the run/config instead of overwriting it.')
    if not path.exists():
        path.write_text(content, encoding='utf-8')

if MODE == 'diagnostic':
    data_root = PROJECT_ROOT / 'data/kaggle_demo'
    if not data_root.exists():
        make_synthetic(data_root, seed=17, samples=24)
    raw = yaml.safe_load((PROJECT_ROOT / 'configs/smoke.yaml').read_text())
    raw['data'].update({s: f'data/kaggle_demo/{s}.jsonl' for s in ('train','val','test')})
else:
    roots = [Path(PREPARED_ROOT)] if PREPARED_ROOT else sorted({
        p.parent for p in INPUT_ROOT.rglob('train.jsonl')
        if (p.parent/'val.jsonl').is_file() and (p.parent/'test.jsonl').is_file()
    })
    if len(roots) != 1:
        raise RuntimeError(f'Set PREPARED_ROOT to the converted release with all three manifests. Found: {roots}')
    source = roots[0]
    raw = yaml.safe_load((PROJECT_ROOT / f'configs/kaggle_{MODE}.yaml').read_text())
    for split in ('train', 'val', 'test'):
        records = read_manifest(source / f'{split}.jsonl')
        content = ''.join(json.dumps(row, sort_keys=True, ensure_ascii=False)+'\n' for row in records)
        relative = f'data/kaggle_real/{split}.jsonl'
        write_stable(PROJECT_ROOT / relative, content)
        raw['data'][split] = relative
raw['train'].update(device=DEVICE, amp=(DEVICE == 'cuda'), output_dir=f'runs/{RUN_NAME}')
cfg = from_dict(raw)
CONFIG_PATH = PROJECT_ROOT / f'configs/kaggle_runtime_{MODE}.yaml'
write_stable(CONFIG_PATH, yaml.safe_dump(raw, sort_keys=False))
print(CONFIG_PATH)
print(yaml.safe_dump(raw, sort_keys=False))


## Validate actual arrays and split separation

This reads all splits. Real-data validation may take considerable I/O and tokenizer loading time. Signer-independent validation is enabled only for the generated diagnostic; official real splits may share signers. Source-group checks still require genuine source metadata.


In [ ]:
from pcat_slt.validate import validate
with torch.no_grad():
    validation_report = validate(cfg, signer_independent=(MODE == 'diagnostic'))
(PROJECT_ROOT / f'reports/kaggle_validation_{MODE}.json').write_text(json.dumps(validation_report, indent=2)+'\n')
print(json.dumps(validation_report, indent=2))


## Train one complete-epoch chunk and measure it

Choose the intended total schedule before starting. STOP_AFTER_EPOCH stops a call without changing that schedule. Epoch-only checkpoints cannot recover unfinished epoch work. CUDA allocation excludes external process memory. This timing includes initialization, input hashing, validation and writes, not just steady-state training steps.


In [ ]:
import time
from pcat_slt.train import train
if DEVICE == 'cuda':
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
checkpoint = train(cfg, resume=str(OUTPUT_DIR/'last.pt') if RESUME else None,
                   stop_after_epoch=STOP_AFTER_EPOCH)
if DEVICE == 'cuda':
    torch.cuda.synchronize()
timing = {'mode': MODE, 'device': DEVICE, 'resumed': RESUME,
          'call_seconds': time.perf_counter()-started, 'stop_after_epoch': STOP_AFTER_EPOCH,
          'planned_total_epochs': cfg.train.epochs,
          'batch_size': cfg.data.batch_size, 'accumulation': cfg.train.accumulation,
          'checkpoint_bytes': Path(checkpoint).stat().st_size,
          'peak_allocated_GiB': torch.cuda.max_memory_allocated()/2**30 if DEVICE=='cuda' else None,
          'peak_reserved_GiB': torch.cuda.max_memory_reserved()/2**30 if DEVICE=='cuda' else None}
(PROJECT_ROOT / f'reports/kaggle_timing_{MODE}_epoch{STOP_AFTER_EPOCH}.json').write_text(json.dumps(timing,indent=2)+'\n')
print(json.dumps(timing,indent=2))
print((OUTPUT_DIR / 'history.jsonl').read_text())


## Generated dev evaluation

The trainer selects best.pt by validation NLL. Generated evaluation is separate and can be expensive; set RUN_GENERATION=False for an initial timing-only call. Test evaluation should happen only after freezing the experiment. Chinese benchmarks need their documented external normalization/scorer.


In [ ]:
from pcat_slt.evaluate import evaluate
if RUN_GENERATION:
    metrics = evaluate(OUTPUT_DIR/'best.pt', manifest=getattr(cfg.data,EVAL_SPLIT), device=DEVICE,
                       output=PROJECT_ROOT/f'evaluation/{RUN_NAME}_{EVAL_SPLIT}')
    print(metrics)


## Bundle outputs and save them before the session ends

This archive contains the run, runtime YAML, reports, predictions and local manifests; diagnostic observations are included for exact recovery. Referenced real arrays, HF caches and third-party corpora are not copied. Download it or verify it appears in a saved notebook version's output. Source and same real-data mount paths are still needed next session.

To continue: attach the extracted saved bundle, keep settings/data unchanged, set RESUME=True and increase STOP_AFTER_EPOCH. Save & Run All may launch a new execution rather than merely snapshot this filesystem; inspect its resulting logs and outputs.


In [ ]:
from IPython.display import FileLink, display

selected = []
selected.extend(p for p in OUTPUT_DIR.rglob('*') if p.is_file())
selected.append(CONFIG_PATH)
selected.extend(p for p in (PROJECT_ROOT/'reports').glob('kaggle_*') if p.is_file())
eval_dir = PROJECT_ROOT/f'evaluation/{RUN_NAME}_{EVAL_SPLIT}'
if eval_dir.is_dir():
    selected.extend(p for p in eval_dir.rglob('*') if p.is_file())
for split in ('train','val','test'):
    selected.append(PROJECT_ROOT / getattr(cfg.data,split))
if MODE == 'diagnostic':
    selected.extend(p for p in (PROJECT_ROOT/'data/kaggle_demo').rglob('*') if p.is_file())
source_manifest = PROJECT_ROOT/'MANIFEST.sha256.json'
if source_manifest.exists():
    selected.append(source_manifest)
selected = sorted(set(selected))
for path in selected:
    if not path.resolve().is_relative_to(PROJECT_ROOT.resolve()):
        raise ValueError('Bundle must contain only project-local artifacts')
raw_bytes = sum(p.stat().st_size for p in selected)
if shutil.disk_usage(PROJECT_ROOT).free < raw_bytes + 100*2**20:
    raise OSError('Insufficient free disk to create the run bundle; save existing checkpoints first')
bundle = PROJECT_ROOT/f'releases/PCAT-SLT-KAGGLE-{MODE}.zip'
checksums = {}
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
    for path in selected:
        relative = path.relative_to(PROJECT_ROOT).as_posix()
        digest = hashlib.sha256()
        with path.open('rb') as f:
            for block in iter(lambda:f.read(1024*1024),b''):
                digest.update(block)
        checksums[relative] = digest.hexdigest()
        z.write(path, 'PCAT-SLT/'+relative)
    z.writestr('PCAT-SLT/RUN_MANIFEST.sha256.json',json.dumps(checksums,indent=2))
print({'bundle':str(bundle),'bytes':bundle.stat().st_size,'files':len(checksums)})
display(FileLink(str(bundle.relative_to(PROJECT_ROOT))))
